In [1]:
!pip install pandas scikit-learn openpyxl joblib streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 37.1 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving Telco_customer_churn.xlsx to Telco_customer_churn.xlsx


In [3]:
import pandas as pd

df = pd.read_excel("Telco_customer_churn.xlsx")
df.head()
df.shape
df.columns.tolist()

['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

df = pd.read_excel("Telco_customer_churn.xlsx")

target = "Churn Value"

drop_cols = ["CustomerID", "Count", "Churn Label", "Churn Score", "CLTV", "Churn Reason"]
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols + [target]).copy()
y = df[target].copy()

X = X.replace([" ", "", "NA", "N/A", "null", "None"], np.nan)
X = X.infer_objects(copy=False)

force_numeric = ["Zip Code", "Latitude", "Longitude", "Tenure Months", "Monthly Charges", "Total Charges"]
force_numeric = [c for c in force_numeric if c in X.columns]

for col in force_numeric:
    X[col] = pd.to_numeric(X[col], errors="coerce")

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

for col in cat_cols:
    X[col] = X[col].astype(str)

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

results = []
best_name = None
best_pipe = None
best_auc = -1

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1 Score": f1_score(y_test, preds),
        "ROC AUC": roc_auc_score(y_test, probs)
    }
    results.append(row)

    if row["ROC AUC"] > best_auc:
        best_auc = row["ROC AUC"]
        best_name = name
        best_pipe = pipe
        best_preds = preds

results_df = pd.DataFrame(results).sort_values("ROC AUC", ascending=False)
print("Best model:", best_name)
print(results_df)
print("Confusion Matrix:")
print(confusion_matrix(y_test, best_preds))

/tmp/ipykernel_26070/2854654035.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.replace([" ", "", "NA", "N/A", "null", "None"], np.nan)


Best model: Gradient Boosting
                 Model  Accuracy  Precision    Recall  F1 Score   ROC AUC
2    Gradient Boosting  0.809794   0.675497  0.545455  0.603550  0.851494
1        Random Forest  0.798439   0.664234  0.486631  0.561728  0.841807
0  Logistic Regression  0.787793   0.616099  0.532086  0.571019  0.833475
Confusion Matrix:
[[937  98]
 [170 204]]
